# Analiza dhe K-Means Clustering

## 1. Ngarkimi i Bibliotekave

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

pd.set_option('display.float_format', '{:.2f}'.format)
sns.set_theme(style='whitegrid', palette='muted')

## 2. Ngarkimi i Dataset-it të Pastruar

In [ ]:
df = pd.read_csv('../data/processed/covid_clean.csv', parse_dates=['date'])
print(f'Shape: {df.shape}')
print(f'Kolonat: {list(df.columns)}')
df.head()

## 3. Statistikat Përshkruese

In [ ]:
STATS_COLS = ['total_cases', 'total_deaths', 'CFR', 'Cases_per_100k',
              'total_vaccinations_per_hundred', 'gdp_per_capita']
df[STATS_COLS].describe().round(2)

## 4. Korrelacioni Pearson — GDP vs Vaksinimi

In [ ]:
from scipy import stats

df_corr = df[['gdp_per_capita', 'total_vaccinations_per_hundred']].dropna()
r, p = stats.pearsonr(df_corr['gdp_per_capita'], df_corr['total_vaccinations_per_hundred'])

fuqi = 'i fortë' if abs(r) > 0.6 else 'i mesëm' if abs(r) > 0.3 else 'i dobët'
dom  = 'domethënës' if p < 0.05 else 'jo domethënës statistikisht'

print(f'Pearson r = {r:.4f}')
print(f'p-value   = {p:.4f}')
print(f'Korrelacion {fuqi}, {dom} (alfa = 0.05)')

## 5. K-Means Clustering — 3 Grupe

**Features e përdorura për grupim:**
- `Cases_per_100k` — intensiteti i pandemisë
- `CFR` — shkalla e vdekshmërisë
- `total_vaccinations_per_hundred` — mbulimi me vaksinë
- `gdp_per_capita` — niveli ekonomik

In [ ]:
CLUSTER_FEATURES = ['Cases_per_100k', 'CFR', 'total_vaccinations_per_hundred', 'gdp_per_capita']

# Hap 1: StandardScaler — normalizim
scaler   = StandardScaler()
X_scaled = scaler.fit_transform(df[CLUSTER_FEATURES])

# Hap 2: KMeans me 3 grupe
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df['Cluster'] = kmeans.fit_predict(X_scaled)

print('Vendet për çdo cluster:')
for c in sorted(df['Cluster'].unique()):
    vendet = df[df['Cluster'] == c]['location'].tolist()
    print(f'  Cluster {c}: {vendet}')

## 6. Interpretimi i Grupeve

Mesataret e çdo cluster na tregojnë profilin e secilit grup.

In [ ]:
# Mesataret e features për çdo cluster
cluster_summary = (
    df.groupby('Cluster')[CLUSTER_FEATURES]
    .mean()
    .round(2)
)
cluster_summary

In [ ]:
# Lista e vendeve për çdo cluster me të gjitha metrikat
for c in sorted(df['Cluster'].unique()):
    subset = df[df['Cluster'] == c][['location', 'Cases_per_100k', 'CFR',
                                      'total_vaccinations_per_hundred', 'gdp_per_capita']]
    print(f'\n── Cluster {c} ({len(subset)} vende) ──')
    print(subset.to_string(index=False))

## 7. Vizualizimi i Cluster-ave (Cases_per_100k vs CFR)

In [ ]:
COLORS = ['#4C72B0', '#C44E52', '#55A868']

fig, ax = plt.subplots(figsize=(12, 7))

for c in sorted(df['Cluster'].unique()):
    sub = df[df['Cluster'] == c]
    ax.scatter(sub['Cases_per_100k'], sub['CFR'],
               color=COLORS[c], s=120, edgecolors='white',
               linewidths=0.8, label=f'Cluster {c}', zorder=3)
    for _, row in sub.iterrows():
        ax.annotate(row['location'],
                    xy=(row['Cases_per_100k'], row['CFR']),
                    xytext=(5, 4), textcoords='offset points',
                    fontsize=7.5, color=COLORS[c])

ax.set_xlabel('Rastet për 100,000 Banorë')
ax.set_ylabel('CFR (%)')
ax.set_title('K-Means Clustering — 20 Vende Evropiane (3 Grupe)')
ax.legend(title='Cluster', fontsize=10)
plt.tight_layout()
plt.show()